# MRI augmentation with MedAugmentX

MR intensities are *arbitrary units*. The same anatomy scanned on two vendors,
or on the same scanner after a coil change, can differ more than healthy and
diseased tissue do. That is what MR augmentation is for: teaching a model to
ignore acquisition variation it will certainly meet, without inventing anatomy
that cannot exist.

This notebook covers the MR-specific transforms, why each one models a real
acquisition effect, and the two properties that make a pipeline trustworthy —
mask alignment and reproducibility.
### Before you start

Everything below runs on **synthetic phantoms** built from analytic shapes —
no patient data, no downloads, no network access. They are illustrations, not
validated physical models, so don't read clinical conclusions off them. The
transform strengths are deliberately exaggerated so the effect is visible in a
single figure; they are *not* recommended training policies.

Install what this notebook needs:

```bash
pip install "medaugmentx[notebooks]"
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import medaugmentx

print("MedAugmentX", medaugmentx.__version__)


def show(*panels, limits=None, cmap="gray", title=None):
    """Display `(label, volume)` pairs side by side on one fixed grey scale.

    A shared scale matters: normalising each panel independently would hide
    exactly the intensity shifts these transforms are meant to introduce.
    """
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
    axes = np.atleast_1d(axes)
    planes = []
    for _, volume in panels:
        image = volume.image if hasattr(volume, "image") else volume
        planes.append(image if image.ndim == 2 else image[image.shape[0] // 2])
    lo, hi = limits if limits else (min(p.min() for p in planes), max(p.max() for p in planes))
    for ax, (label, _), plane in zip(axes, panels, planes):
        ax.imshow(plane, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
        ax.set_title(label, fontsize=11)
        ax.set_axis_off()
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


## 1. A synthetic brain slice

`medaugmentx.phantoms` ships four modality phantoms. They are generated, not
loaded, so this notebook is reproducible anywhere.


In [ ]:
from medaugmentx.phantoms import mri_phantom

volume = mri_phantom()
print("shape   ", volume.shape)
print("spacing ", volume.spacing)
print("modality", volume.modality)
print("range   ", (float(volume.image.min()), float(volume.image.max())))

show(("Synthetic MR slice", volume), limits=(0, 1))

## 2. Rician noise, not Gaussian

This is the MR-specific detail general-purpose libraries get wrong. MR
magnitude images are the modulus of a complex signal, so the noise floor is
**Rician**: in background air, where true signal is ~0, the noise is strictly
positive and biases the mean *upward*. Gaussian noise would let background go
negative, which no real magnitude image does.

Watch the background statistics, not just the picture.


In [ ]:
from medaugmentx.transforms import RicianNoise

noisy = RicianNoise(std=0.09, seed=7)(volume)

background = volume.image < 0.01   # air outside the head
print(f"background mean  before {volume.image[background].mean():.4f}"
      f"   after {noisy.image[background].mean():.4f}")
print(f"background min   before {volume.image[background].min():.4f}"
      f"   after {noisy.image[background].min():.4f}   <- stays >= 0")

show(("Original", volume), ("RicianNoise(std=0.09)", noisy), limits=(0, 1))

## 3. Intensity non-uniformity (bias field)

A smooth multiplicative field across the image, caused by coil sensitivity.
It is the classic reason a model trained on one scanner degrades on another:
the same tissue reads brighter on one side of the image than the other.

`alpha` sets the strength; `coarse_shape` sets how many control points the
field is built from — smaller means smoother.


In [ ]:
from medaugmentx.transforms import BiasField

gentle = BiasField(alpha=0.25, coarse_shape=3, seed=7)(volume)
strong = BiasField(alpha=0.65, coarse_shape=3, seed=7)(volume)

show(("Original", volume), ("alpha=0.25", gentle), ("alpha=0.65", strong), limits=(0, 1))

## 4. k-space artifacts: ghosting and dropout

MR is acquired in the frequency domain, so some artifacts are only sensible
there. Both of these transforms move the image into k-space, corrupt it, and
transform back — which is why the result looks nothing like adding noise in
image space.

- **`GhostingArtifact`** — periodic motion (breathing, pulsatile flow) replicates
  the anatomy along the phase-encode axis.
- **`KSpaceDropout`** — dropped readout lines, producing structured ripple.


In [ ]:
from medaugmentx.transforms import GhostingArtifact, KSpaceDropout

ghosted = GhostingArtifact(ghost_intensity=0.18, ghost_shift=28, num_ghosts=2, seed=7)(volume)
dropped = KSpaceDropout(dropout_fraction=0.06, seed=7)(volume)

show(("Original", volume), ("Ghosting", ghosted), ("k-space dropout", dropped), limits=(0, 1))

## 5. Patient motion

`MRIMotion` composes several small rigid movements during the acquisition,
producing the directional blur-and-double-edge that a restless patient causes.


In [ ]:
from medaugmentx.transforms import MRIMotion

moved = MRIMotion(degrees=4.0, translation=3.0, num_movements=3, seed=7)(volume)
show(("Original", volume), ("MRIMotion", moved), limits=(0, 1))

## 6. Masks stay aligned with the image

For segmentation this is the property that matters most. A spatial transform
must move the mask by the *same* sampled parameters as the image, and must
interpolate it with nearest-neighbour so label values are never averaged into
nonexistent classes.

MedAugmentX does both. Here the mask has labels 1 and 2; after a rotation the
label set is unchanged — no invented `1.5`.


In [ ]:
from medaugmentx import MedVolume
from medaugmentx.transforms import RandomAffine

mask = np.zeros(volume.shape, dtype=np.uint8)
mask[volume.image > 0.55] = 1                       # bright tissue
mask[volume.image > 0.70] = 2                       # brightest tissue
labelled = MedVolume(volume.image, mask=mask, metadata=volume.metadata)

rotated = RandomAffine(rotation=15.0, seed=3)(labelled)

print("labels before:", np.unique(labelled.mask))
print("labels after :", np.unique(rotated.mask), " <- no interpolated labels")
print("mask dtype   :", rotated.mask.dtype)

show(("Image (rotated)", rotated), ("Mask (rotated)", MedVolume(rotated.mask.astype(float))))

## 7. Reproducibility

An augmented result you cannot regenerate is not a result. Every transform
takes a `seed`, and the same seed gives byte-identical output — on any machine,
in any process.


In [ ]:
a = RicianNoise(std=0.05, seed=42)(volume)
b = RicianNoise(std=0.05, seed=42)(volume)
c = RicianNoise(std=0.05, seed=43)(volume)

print("same seed  -> identical:", np.array_equal(a.image, b.image))
print("other seed -> identical:", np.array_equal(a.image, c.image))

## 8. A preset pipeline, and reading it back

`mri_pipeline()` composes a sensible MR starting point. `pipeline_summary()`
prints exactly what will be applied — worth pasting into an experiment log,
since "we used standard augmentation" is not a reproducible statement.


In [ ]:
from medaugmentx import pipeline_summary
from medaugmentx.presets import mri_pipeline

pipeline = mri_pipeline(seed=0)
print(pipeline_summary(pipeline))

augmented = pipeline(volume)
show(("Original", volume), ("mri_pipeline(seed=0)", augmented), limits=(0, 1))

## 9. Save the policy with the experiment

Serialise the pipeline to JSON and store it next to your checkpoints. Reloading
reconstructs the same transforms, so a reviewer can reproduce the augmentation
a year later.


In [ ]:
from medaugmentx.serialization import from_json, to_json

policy = to_json(pipeline)
print(policy[:280], "...")

restored = from_json(policy)
print("\nround-trips:", pipeline_summary(restored) == pipeline_summary(pipeline))

## Where to go next

- **[`docs/API_REFERENCE.md`](../docs/API_REFERENCE.md)** — every public import.
- **[`docs/RESEARCH_GUIDE.md`](../docs/RESEARCH_GUIDE.md)** — replay, worker
  seeds, and how to report augmentation in a paper.
- **[`examples/`](../examples/)** — runnable scripts, including
  `safe_augmentation.py` (validator + `Guard`) and `keypoints_bboxes.py`.
- **The other tutorials** — `01_mri_augmentation.ipynb`,
  `02_ct_augmentation.ipynb`, `03_dbt_augmentation.ipynb`.

Choosing augmentation strength is an empirical question for *your* dataset and
task. Start weak, look at the images, and validate against a held-out set.
